# 02 – Reactive transport: Acid Mine Drainage from the tailings storage facility

In the previous notebook we transported a conservative tracer from the TSF to understand advection and dispersion. Now we ask: **what happens to the water chemistry?**

Acid Mine Drainage (AMD) forms when sulphide minerals — mainly **pyrite (FeS₂)** — oxidise on contact with oxygen and water:

$$\text{FeS}_2 + \tfrac{15}{4}\text{O}_2 + \tfrac{7}{2}\text{H}_2\text{O} \rightarrow \text{Fe}(\text{OH})_3 + 2\,\text{SO}_4^{2-} + 4\,\text{H}^+$$

AMD from the TSF is highly acidic (pH ≈ 4) and enriched in Fe, Al, and SO₄. As it migrates through the aquifer it reacts with the matrix: dissolving carbonates, precipitating ferrihydrite, and acidifying groundwater en route to the groundwater-dependent ecosystem (GDE).

## What `mf6rtm` does

`mf6rtm` couples **MODFLOW 6** (flow + transport) with **PHREEQC** (geochemistry) via the ModflowAPI BMI. At each transport time step, GWT concentrations are passed to PHREEQC, which runs geochemical reactions and returns updated concentrations. One reactive GWT model is created per PHREEQC component, all driven by the same GWF flow field.

In [ ]:
import os
import shutil
import pyemu
import flopy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mf6rtm import utils, mup3d
from herebedragons import prep_bins, specify_tsf_cells, plot_setup, sout_to_array

ws          = 'model/flow_transport'   # GWF + tracer GWT built in notebooks 00 and 01
reactive_ws = 'model/reactive'         # mf6rtm will write the reactive simulation here


if os.path.exists(reactive_ws):
    shutil.rmtree(reactive_ws)
os.makedirs(reactive_ws)

## Load the GWF + conservative-tracer GWT simulation

We load the simulation built in notebooks 00 and 01.  
`mf6rtm` uses the tracer GWT as a **template**: it inherits the grid, solver settings, dispersion, and porosity, then clones it into one reactive GWT model per PHREEQC component.

> Make sure you have run notebooks `00` and `01` at least once so the model files exist in `model/flow_transport/`.

In [ ]:
sim      = flopy.mf6.MFSimulation.load(sim_ws=ws, verbosity_level=0)
gwf      = sim.get_model('gwf')
gwt_name = 'gwt'   # conservative tracer model from notebook 01

pitcells = np.argwhere(gwf.dis.idomain.array[0] == 2)
print('Models in sim:', sim.model_names)
print('Grid:         ', gwf.dis.nlay.data, 'layers ×',
      gwf.dis.nrow.data, 'rows ×', gwf.dis.ncol.data, 'cols')

## Defining the water chemistry

We define **two PHREEQC solutions** that represent the two end-members:

| # | Name | pH | SO₄ (mol/L) | Fe (mol/L) | Description |
|---|------|----|-------------|------------|-------------|
| 1 | Aquifer background | 6.96 | 7.5 × 10⁻³ | 5 × 10⁻⁵ | Neutral, mildly mineralised |
| 2 | Tailings leachate (AMD) | 3.99 | 5.0 × 10⁻² | 3 × 10⁻² | Acidic, high SO₄ & Fe |

Concentrations are in **mol/kg water** (PHREEQC default units).  
Solution 1 is assigned as the initial condition everywhere (`set_ic(1)`).  
Solution 2 will be injected at the TSF boundary (CNC package).

In [ ]:
solutionsdf = pd.DataFrame({
    'aquifer': {
        'pH':    6.96,    'pe':    1.67,
        'C(+4)': 3.94e-3, 'S(6)':  7.48e-3,
        'Fe(+2)':5.39e-5, 'Fe(+3)':2.32e-8,
        # 'Mn(+2)':4.73e-5, 
        'Ca':    6.92e-3,
        'Mg':    1.96e-3, 'Na':    1.30e-3,
        'K':     6.65e-5, 'Cl':    1.03e-3,
        'Al':    1.27e-7, 
        # 'Si':    1.94e-3,
    },
    'tailings': {
        'pH':    3.99,    'pe':    7.69,
        'C(+4)': 4.92e-4, 'S(6)':  5.00e-2,
        'Fe(+2)':3.06e-2, 'Fe(+3)':1.99e-7,
        # 'Mn(+2)':9.83e-6, 
        'Ca':    1.08e-2,
        'Mg':    9.69e-4, 'Na':    1.39e-3,
        'K':     7.93e-4, 'Cl':    1.19e-4,
        'Al':    4.30e-3, 
        # 'Si':    2.08e-3,
    },
    'mar': {
        'pH':    6.80,    'pe':    10.5,   # near atmospheric O2
        'C(+4)': 1.80e-3, 'S(6)':  2.00e-4,
        'Fe(+2)':1.00e-7, 'Fe(+3)':1.00e-8,
        # 'Mn(+2)':1.00e-7,
        'Ca':    8.00e-4,
        'Mg':    3.00e-4, 'Na':    1.50e-3,
        'K':     5.00e-5, 'Cl':    1.40e-3,
        'Al':    1.00e-8, 
        # 'Si':    3.00e-4,
    },
})

# solutionsdf = pd.DataFrame({
#     'aquifer': {
#         'pH':    6.96,    'pe':    1.67,
#         'C(+4)': 3.94e-30, 'S(6)':  7.48e-3,
#         'Fe(+2)':5.39e-5, 'Fe(+3)':2.32e-8,
#         'Mn(+2)':4.73e-5, 
#         'Ca':    6.92e-3,
#         'Mg':    1.96e-3, 'Na':    1.30e-3,
#         'K':     6.65e-5, 'Cl':    1.03e-3,
#         'Al':    1.27e-7, 'Si':    1.94e-3,
#     },
#     'tailings': {
#         'pH':    6.96,    'pe':    1.67,
#         'C(+4)': 3.94e-30, 'S(6)':  7.48e-3,
#         'Fe(+2)':5.39e-5, 'Fe(+3)':2.32e-8,
#         'Mn(+2)':4.73e-1, 
#         'Ca':    6.92e-3,
#         'Mg':    1.96e-3, 'Na':    1.30e-3,
#         'K':     6.65e-5, 'Cl':    1.03e-3,
#         'Al':    1.27e-7, 'Si':    1.94e-3,
#     },
# })
solutionsdf.index.name = 'component'
display(solutionsdf)

solutions = mup3d.Solutions(utils.solution_df_to_dict(solutionsdf))
solutions.set_ic(1)   # every cell starts with background aquifer chemistry (solution 1)

## Aquifer mineralogy: equilibrium with Calcite

As AMD migrates from the TSF, the aquifer's **Acid Neutralising Capacity (ANC)** determines
whether the plume reaches the GDE or is attenuated first.

The primary ANC mineral is **Calcite** (CaCO₃) — the fastest-dissolving carbonate:

$$\text{CaCO}_3 + 2\,\text{H}^+ \rightarrow \text{Ca}^{2+} + \text{H}_2\text{O} + \text{CO}_2$$

This reaction consumes H⁺ from the AMD, raising pH and precipitating dissolved metals (Fe, Al).
Once the Calcite reserve is exhausted, the pH front breaks through and acidification reaches the receptor.

The table below lists the full mineral assemblage; we keep **Calcite only** for simplicity.
`conc_mol_l` is the initial content in mol/L bulk volume, converted to mol/L pore water via porosity.

In [ ]:
prsity = 0.3   # effective porosity (must match MST package in GWT)

eq_df = pd.DataFrame([
    {'phase': 'Calcite',    'sat_index': 0.0, 'conc_mol_l': 1.95e-4, 'num': 1},
    {'phase': 'Siderite',   'sat_index': 0.0, 'conc_mol_l': 4.22e-3, 'num': 1},
    {'phase': 'Gibbsite',   'sat_index': 0.0, 'conc_mol_l': 2.51e-3, 'num': 1},
    {'phase': 'Fe(OH)3(a)', 'sat_index': 0.0, 'conc_mol_l': 1.86e-3, 'num': 1},
    {'phase': 'Gypsum',     'sat_index': 0.0, 'conc_mol_l': 0.0,     'num': 1},
    {'phase': 'SiO2(a)',    'sat_index': 0.0, 'conc_mol_l': 4.07e-1, 'num': 1},
])

# Keep Calcite only — primary ANC mineral for AMD buffering
eq_df = eq_df[eq_df['phase'].isin(['Calcite', 
                                #    'Fe(OH)3(a)'
                                   ])].reset_index(drop=True)

eq_df['conc_mol_lb'] = [
    utils.concentration_volbulk_to_volwater(v, prsity)
    for v in eq_df['conc_mol_l'].values
]

equilibrium_phases = utils.parse_equilibriums_dataframe(eq_df)
eq_phases = mup3d.EquilibriumPhases(equilibrium_phases)
eq_phases.set_ic(1)   # zone 1 everywhere

display(eq_df[['phase', 'sat_index', 'conc_mol_l', 'conc_mol_lb']])
print('\nEquilibrium phases:', eq_phases.names)

## Create the reactive transport model

`Mup3d.from_mf6()` reads the GWF + GWT simulation and creates the reactive model object. The tracer GWT is used as a template — grid, solver, dispersion, and porosity are inherited and cloned into one reactive GWT per PHREEQC component.

In [ ]:
model = mup3d.Mup3d.from_mf6(sim, solutions, name='rtm', gwt_name=gwt_name)
model.set_wd(reactive_ws)   # override the default path (same in practice; explicit is better)

print('Grid:        ', model.nlay, model.nrow, model.ncol, f'({model.nxyz:,} cells)')
print('Working dir: ', model.wd)

## Assign mineral phases to the model

`set_equilibrium_phases()` validates each mineral name against the database and registers the initial condition zones. Zone 1 covers the entire domain.

In [ ]:
model.set_equilibrium_phases(eq_phases)
# print('Equilibrium phases registered:', model.equilibrium_phases.names)

## PHREEQC thermodynamic database

The database contains equilibrium constants for aqueous species and minerals, plus RATES blocks for kinetically controlled reactions. `set_database()` copies it into the working directory so the simulation is self-contained.

In [ ]:
database = os.path.join('..', 'asset', 'phreeqc.dat')
model.set_database(database)
print('Database copied from:', model.database)

In [ ]:
postfix = os.path.join('..', 'asset', 'postfix.dat')
model.set_postfix(postfix)

## Initialise PHREEQC

`model.initialize()` generates the PHREEQC input script, creates the PhreeqcRM object, and equilibrates all grid cells to solution 1 (aquifer background). Initial concentrations are converted to mol/m³ arrays for MODFLOW 6 GWT. Component names are stored in `model.components` — these become the names of the reactive GWT models.

This step may take a minute on the full grid.

In [ ]:
# model.set_charge_offset(-11)
model.set_componenth2o(True)

In [ ]:
model.initialize(add_charge_flag='Ca')
print('PHREEQC components:', model.components)

## TSF chemical stress — injecting AMD at the boundary

The tailings storage facility is represented as a **constant-concentration** (CNC) boundary:

- The same 15 TSF footprint cells used in notebook 01.
- Each cell is assigned **solution 2** (tailings AMD chemistry).
- `model.set_chem_stress()` equilibrates the AMD chemistry through PHREEQC and stores per-component concentrations for the CNC packages in each reactive GWT model.

In [ ]:
tsf_cells_raw = specify_tsf_cells()               # returns [((0, row, col), conc), ...]
tsf_cellids   = [item[0] for item in tsf_cells_raw]  # just the (layer, row, col) tuples

cs = mup3d.ChemStress('tsf', type='cnc')
cs.set_spd([2] * len(tsf_cellids))   # solution 2 = tailings AMD for every TSF cell
cs.set_cells(tsf_cellids)

model.set_chem_stress(cs)  # equilibrates TSF chemistry and stores per-component concentrations
print(f'TSF: {len(tsf_cellids)} cells, solution 2 (tailings AMD)')

In [ ]:
# GHB eastern boundary: inject background chemistry (solution 1) for all inflow cells.
# _update_gwt_stress_packages() strips the 'tracer' aux and replaces with component concs.
ghb_pkg = gwf.get_package('ghb-inflow')
n_ghb   = len(ghb_pkg.stress_period_data.get_data()[0])   # 100 cells

cs_ghb = mup3d.ChemStress('ghb-inflow', type='aux')
cs_ghb.set_spd([1] * n_ghb)   # solution 1 = background aquifer chemistry
model.set_chem_stress(cs_ghb)
print(f'GHB: {n_ghb} cells, solution 1 (background)')

In [ ]:
# GHB eastern boundary: inject background chemistry (solution 1) for all inflow cells.
# _update_gwt_stress_packages() strips the 'tracer' aux and replaces with component concs.
wel_mar_pkg = gwf.get_package('wel-mar')
n_mar   = len(wel_mar_pkg.stress_period_data.get_data()[1])   # 100 cells

cs_mar = mup3d.ChemStress('wel-mar', type='aux')
cs_mar.set_spd([3] * n_mar)   # solution 1 = background aquifer chemistry
model.set_chem_stress(cs_mar)
print(f'WEL-MAR: {n_mar} cells, solution 1 (background)')

In [ ]:
rcha_pkg = gwf.get_package('rcha')
n_rcha   = len(rcha_pkg.recharge.get_data()[0]) 

cs_rcha = mup3d.ChemStress('rcha_0', type='aux')
cs_rcha.set_spd([1])   # solution 1 = background aquifer chemistry
model.set_chem_stress(cs_rcha)
print(f'RECHARGE: {n_rcha} cells, solution 1 (background)')

## Write and run the reactive simulation

`model.write_simulation()` assembles the final simulation: clones the tracer GWT into N reactive GWT models (one per PHREEQC component), assigns component-specific initial conditions and CNC packages, and writes all MODFLOW 6 and PHREEQC input files.

`model.run()` launches the coupled simulation via the MODFLOW 6 BMI and PhreeqcRM.

In [ ]:
# Copy the MODFLOW 6 shared library — mf6rtm uses ModflowAPI (BMI), not the mf6 executable
prep_bins(reactive_ws, get_only=['libmf6', 'mf6'])

# Pit + dewatering-ring cells desaturate so hard during dewatering that PHREEQC
# cannot converge them. Exclude them from reactions: MODFLOW still transports
# them, but PHREEQC skips them (their saturation is forced to 0 each reaction step).
ncol = int(gwf.dis.ncol.data)
no_react = set()
for (r, c) in np.argwhere(gwf.dis.idomain.array[0] == 2):                     # pit
    no_react.add(int(r) * ncol + int(c))
for (_, r, c) in gwf.get_package('wel-dewater').stress_period_data.get_data()[1]['cellid']:
    no_react.add(int(r) * ncol + int(c))                                      # dewatering ring
no_react = sorted(no_react)
print(f'{len(no_react)} cells excluded from reactions (pit + dewatering ring)')

model.set_config(
    reactive={
        "timing": 'all',      # this can be user or all
        "externalio": True,   # flag to write external files
    },
    solver={
        "min_concentration": 1e-12,
        "no_react_cells": no_react,
    },
)
model.write_simulation()

In [ ]:
pyemu.os_utils.run('mf6rtm', cwd=reactive_ws)

## Visualise the results

We plot spatial maps of **Cl** and **pH** at each output timestep.

- **Cl** tracks where AMD-origin water has migrated — it is nearly conservative at these pH values and arrives ahead of the acid front.
- **pH** shows where the aquifer's carbonate alkalinity has been depleted enough for the acid front to break through.

In [ ]:
import matplotlib.patches as patches

sout      = pd.read_csv(os.path.join(reactive_ws, 'sout.csv'))
times     = sorted(sout['time_d'].unique())
labels    = [f'{t/365:.0f} yr' for t in times]
mg        = gwf.modelgrid
idom      = gwf.dis.idomain.array[0]
cell_size = float(gwf.dis.delr.get_data()[0])

tsf_ids = [c[0] for c in specify_tsf_cells()]
tsf_x   = [mg.xcellcenters[r, c] for (_, r, c) in tsf_ids]
tsf_y   = [mg.ycellcenters[r, c] for (_, r, c) in tsf_ids]
tsf_rect_kw = dict(
    xy=(min(tsf_x) - cell_size/2, min(tsf_y) - cell_size/2),
    width=max(tsf_x) - min(tsf_x) + cell_size,
    height=max(tsf_y) - min(tsf_y) + cell_size,
    lw=1.5, edgecolor='navy', facecolor='steelblue', alpha=0.4,
)

# Pit rectangle from cell centres
pit_x = [mg.xcellcenters[r, c] for r, c in pitcells]
pit_y = [mg.ycellcenters[r, c] for r, c in pitcells]
pit_rect_kw = dict(
    xy=(min(pit_x) - cell_size/2, min(pit_y) - cell_size/2),
    width=max(pit_x) - min(pit_x) + cell_size,
    height=max(pit_y) - min(pit_y) + cell_size,
    lw=1.2, edgecolor='k', facecolor='none',
)

In [ ]:
# End-of-stress-period times from TDIS → closest available sout output time
perlen   = [p[0] for p in sim.tdis.perioddata.get_data()]
sp_ends  = np.cumsum(perlen)

times_all = np.array(sorted(sout['time_d'].unique()))
sp_times  = [float(times_all[np.argmin(np.abs(times_all - t))]) for t in sp_ends]
sp_labels = [f'SP{i+1} ({t/365:.0f} yr)' for i, t in enumerate(sp_times)]

vars_cfg = [
    ('solution_total_molality_Fe', 'Cl', 'RdYlBu_r', (None, None)),
    # ('solution_ph', 'pH', 'RdYlBu', (3, 8)),
]

for col_name, label, cmap, (vmin, vmax) in vars_cfg:
    arrays = [sout_to_array(sout[sout['time_d'] == t], col_name, idom) for t in sp_times]
    if vmin is None:
        vals = np.concatenate([a[np.isfinite(a)].ravel() for a in arrays])
        vmin, vmax = float(vals.min()), float(vals.max())

    fig, axes = plt.subplots(1, len(sp_times), figsize=(4 * len(sp_times), 4), constrained_layout=True)
    fig.suptitle(label, fontsize=20, fontweight='bold')
    for ax, arr, lbl in zip(axes, arrays, sp_labels):
        mapview = flopy.plot.PlotMapView(model=gwf, layer=0, ax=ax)
        pc = mapview.plot_array(arr,
                                # vmin=0, vmax=vmax,
                                cmap=cmap, alpha=0.85)
        mapview.plot_grid(alpha=0.2)
        ax.add_patch(patches.Rectangle(**pit_rect_kw))
        # ax.add_patch(patches.Rectangle(**tsf_rect_kw))
        mapview.plot_bc('wel-mar', color='green', zorder=100, kper=2)
        ax.set_title(lbl)
        ax.set_aspect('equal')
        ax.set_xticks([])
        ax.set_yticks([])
    fig.colorbar(pc, ax=ax, shrink=0.7, label=label)
    plt.show()



In [ ]:

# (skip this block if already defined by the Cl cell)
perlen   = [p[0] for p in sim.tdis.perioddata.get_data()]
sp_ends  = np.cumsum(perlen)
times_all = np.array(sorted(sout['time_d'].unique()))
sp_times  = [float(times_all[np.argmin(np.abs(times_all - t))]) for t in sp_ends]
sp_labels = [f'SP{i+1} ({t/365:.0f} yr)' for i, t in enumerate(sp_times)]

vars_cfg = [
    # ('solution_total_molality_Mn', 'pH', 'RdYlBu_r', (None, None)),
    ('solution_ph', 'pH', 'RdYlBu', (3, 8)),
]

for col_name, label, cmap, (vmin, vmax) in vars_cfg:
    arrays = [sout_to_array(sout[sout['time_d'] == t], col_name, idom) for t in sp_times]
    if vmin is None:
        vals = np.concatenate([a[np.isfinite(a)].ravel() for a in arrays])
        vmin, vmax = float(vals.min()), float(vals.max())

    fig, axes = plt.subplots(1, len(sp_times), figsize=(4 * len(sp_times), 4), constrained_layout=True)
    fig.suptitle(label, fontsize=20, fontweight='bold')
    for ax, arr, lbl in zip(axes, arrays, sp_labels):
        mapview = flopy.plot.PlotMapView(model=gwf, layer=0, ax=ax)
        pc = mapview.plot_array(arr,
                                vmin=vmin, vmax=vmax,
                                cmap=cmap, alpha=0.85)
        mapview.plot_grid(alpha=0.2)
        ax.add_patch(patches.Rectangle(**pit_rect_kw))
        ax.add_patch(patches.Rectangle(**tsf_rect_kw))
        mapview.plot_bc('wel-mar', color='red', zorder=10, kper=2)
        ax.set_title(lbl)
        ax.set_aspect('equal')
        ax.set_xticks([])
        ax.set_yticks([])
    fig.colorbar(pc, ax=ax, shrink=0.7, label=label)
    plt.show()


In [ ]:
# # UCN vs sout Cl — unit conversion and element-wise comparison
# #
# # Exact back-conversion:  ucn_molkg = UCN × sat / (1000 × water_mass)  =  UCN / (1000 × ρ)
# # where ρ = water_mass / sat  [kg/L] ≈ 0.991–1.001 for this aquifer.

# idom   = gwf.dis.idomain.array[0]
# active = idom >= 1
# top    = gwf.dis.top.array
# botm   = gwf.dis.botm.array[0]

# sout_cl   = pd.read_csv(os.path.join(reactive_ws, 'sout.csv'))
# sout_ts   = sorted(sout_cl['time_d'].unique())
# hds       = flopy.utils.HeadFile(os.path.join(reactive_ws, 'gwf.hds'))
# ucn_cl    = flopy.utils.HeadFile(os.path.join(reactive_ws, 'Cl.ucn'), text='CONCENTRATION')
# kstpkpers = ucn_cl.get_kstpkper()

# fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
# fig.suptitle('UCN − sout  [mol/kgw]  (should be ≈ 0 everywhere)', fontsize=11)

# for ax, ksp, lbl, t in zip(axes, kstpkpers, labels, sout_ts):
#     head  = hds.get_data(kstpkper=ksp)[0]
#     sat   = np.clip((head - botm) / (top - botm), 0, 1)

#     sout_t   = sout_cl[sout_cl['time_d'] == t]
#     wmass    = sout_to_array(sout_t, 'solution_water_mass', idom)          # kg
#     sout_arr = sout_to_array(sout_t, 'solution_total_molality_Cl', idom)   # mol/kgw

#     ucn_arr   = ucn_cl.get_data(kstpkper=ksp)[0]           # mol/m³
#     ucn_molkg = ucn_arr * sat / (1000.0 * wmass)            # mol/kgw  (÷ ρ)
#     diff = ucn_molkg - sout_arr
#     diff[idom <= 0] = np.nan

#     mapview = flopy.plot.PlotMapView(model=gwf, layer=0, ax=ax)
#     pc = mapview.plot_array(np.round(diff, 8), cmap='RdBu', alpha=0.85)
#     mapview.plot_grid(alpha=0.2)
#     ax.add_patch(patches.Rectangle(**pit_rect_kw))
#     ax.add_patch(patches.Rectangle(**tsf_rect_kw))
#     ax.set_title(lbl)
#     ax.set_aspect('equal')
#     ax.set_xticks([])
#     ax.set_yticks([])

# fig.colorbar(pc, ax=list(axes), label='UCN − sout  [mol/kgw]', shrink=0.7)
# plt.show()

In [ ]:
import matplotlib.patches as patches

ucn_file = os.path.join(reactive_ws, 'Cl.ucn')
cobj     = flopy.utils.HeadFile(ucn_file, text='CONCENTRATION')
kstpkper = np.array(cobj.get_kstpkper())
labels   = [f'{t/365:.0f} yr' for t in cobj.get_times()]

c0_arr = cobj.get_data(kstpkper=tuple(kstpkper[0]))[0].astype(float)
c0_arr[idom <= 0] = np.nan
c0 = float(np.nanmin(c0_arr))

fig, axes = plt.subplots(1, len(kstpkper), figsize=(5*len(kstpkper), 4), constrained_layout=True)
fig.suptitle(f'Cl  (C₀ − C,  C₀ = {c0:.3f} mol/m³)', fontsize=11, fontweight='bold')

absmax = c0
for ax, ksp, lbl in zip(axes, kstpkper, labels):
    data = c0_arr - cobj.get_data(kstpkper=tuple(ksp))[0].astype(float) 
    data[idom <= 0] = np.nan

    mapview = flopy.plot.PlotMapView(model=gwf, layer=0, ax=ax)
    pc = mapview.plot_array(data,  vmin=0, vmax= None, cmap='YlOrRd', alpha=0.85)
    mapview.plot_grid(alpha=0.05)
    ax.add_patch(patches.Rectangle(**pit_rect_kw))
    ax.add_patch(patches.Rectangle(**tsf_rect_kw))
    mapview.plot_bc("wel-mar",  kper=1)
    ax.set_title(lbl)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])

fig.colorbar(pc, ax=list(axes), label='Cl  C₀ − C  (mol/m³)', shrink=0.7)
plt.show()


## Management quantity: pH at the GDE

The management criterion for this project is: **keep pH at the groundwater-dependent ecosystem above 5 at all times**.

We track the **minimum pH across the GDE drain footprint** at each output time. This is equivalent to the compliance statement — if the minimum stays above 5, every drain cell does — and it is robust to shifts in the plume path when parameters change.

`proc_ph_at_gde()` also writes the series to `gde_ph.csv` in the model folder, so it can serve directly as a PEST observation file (one observation per output time). The worst case over time is the single number a manager would track.

In [ ]:
from herebedragons import proc_ph_at_gde

_, ph_gde = proc_ph_at_gde(wd=reactive_ws)   # also writes model/reactive/gde_ph.csv
display(ph_gde)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ph_gde.index / 365, ph_gde['ph_min'], 'o-', color='tab:blue', label='min pH at GDE so animals don\'t die')
ax.axhline(5.5, color='crimson', ls='--', label='management criterion (pH 5)')
ax.set_xlabel('time (years)')
ax.set_ylabel('pH')
ax.set_title('Minimum pH across the GDE drain cells')
ax.legend()
plt.show()

worst = ph_gde['ph_min'].min()
print(f"Management quantity — worst pH at GDE over the simulation: {worst:.2f} "
      f"({'OK' if worst > 5.5 else 'CRITERION BREACHED'})")